In [1]:
import sys

In [2]:
%load_ext sql
%sql sqlite:///clasher.db
%load_ext memory_profiler

In [3]:
%%time
%%memit
import csv
import sqlite3
    
conn = sqlite3.connect("clasher.db")
cursor = conn.cursor()

with open('battlesStaging_12072020_to_12262020_WL_tagged.csv', 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    columns = reader.fieldnames
    
    # Create table
    col_defs = ", ".join([f'"{col}" TEXT' for col in columns])
    cursor.execute(f"DROP TABLE IF EXISTS clasher")
    cursor.execute(f"CREATE TABLE clasher ({col_defs})")
    
    # Insert all rows
    placeholders = ", ".join(["?" for _ in columns])
    rows = [tuple(row[col] for col in columns) for row in reader]
    
    for row in rows:
        cursor.execute(f'INSERT INTO clasher VALUES ({placeholders})', row)

    conn.commit()

print("Done! Rows loaded:", cursor.execute("SELECT COUNT(*) FROM clasher").fetchone()[0])
print("First row:", cursor.execute("SELECT * FROM clasher").fetchone())

Done! Rows loaded: 16795959
First row: ('0', '2020-12-07 07:00:00+00:00', '54000049.0', '72000201.0', '6590.0', '#28RR8PJP0', '6581.0', '31.0', '2.0', '4768.0', '[1218]', '#YYL2GJUC', '16000111.0', '#9UU8P008P', '6599.0', '-31.0', '1.0', '147.0', '#P2PUU9JV', '16000119.0', '', '', '26000036', '13', '28000015', '13', '26000050', '13', '26000044', '13', '26000054', '13', '28000016', '13', '26000043', '13', '26000062', '13', '[26000036, 26000043, 26000044, 26000050, 26000054, 26000062, 28000015, 28000016]', '104', '7', '0', '1', '1', '2', '3', '2', '3.625', '28000004', '13', '26000041', '13', '26000026', '13', '26000030', '13', '26000000', '13', '28000011', '13', '28000003', '13', '27000006', '13', '[26000000, 26000026, 26000030, 26000041, 27000006, 28000003, 28000004, 28000011]', '104', '4', '1', '3', '4', '1', '1', '2', '3.125')
peak memory: 27167.28 MiB, increment: 27032.44 MiB
CPU times: total: 16min 49s
Wall time: 20min 44s


In [4]:
cursor.execute("SELECT * FROM clasher")
print([desc[0] for desc in cursor.description])

['', 'battleTime', 'arena.id', 'gameMode.id', 'average.startingTrophies', 'winner.tag', 'winner.startingTrophies', 'winner.trophyChange', 'winner.crowns', 'winner.kingTowerHitPoints', 'winner.princessTowersHitPoints', 'winner.clan.tag', 'winner.clan.badgeId', 'loser.tag', 'loser.startingTrophies', 'loser.trophyChange', 'loser.crowns', 'loser.kingTowerHitPoints', 'loser.clan.tag', 'loser.clan.badgeId', 'loser.princessTowersHitPoints', 'tournamentTag', 'winner.card1.id', 'winner.card1.level', 'winner.card2.id', 'winner.card2.level', 'winner.card3.id', 'winner.card3.level', 'winner.card4.id', 'winner.card4.level', 'winner.card5.id', 'winner.card5.level', 'winner.card6.id', 'winner.card6.level', 'winner.card7.id', 'winner.card7.level', 'winner.card8.id', 'winner.card8.level', 'winner.cards.list', 'winner.totalcard.level', 'winner.troop.count', 'winner.structure.count', 'winner.spell.count', 'winner.common.count', 'winner.rare.count', 'winner.epic.count', 'winner.legendary.count', 'winner.e

In [5]:
%%time
%%memit
with open('CardMasterListSeason18_12082020.csv', 'r', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    columns = reader.fieldnames
    
    # Create table
    col_defs = ", ".join([f'"{col}" TEXT' for col in columns])
    cursor.execute("DROP TABLE IF EXISTS card_names")
    cursor.execute(f"CREATE TABLE card_names ({col_defs})")
    
    # Insert all rows
    placeholders = ", ".join(["?" for _ in columns])
    rows = [tuple(row[col] for col in columns) for row in reader]
    
    for row in rows:
        cursor.execute(f'INSERT INTO card_names VALUES ({placeholders})', row)
    conn.commit()

print("Done! Rows loaded:", cursor.execute("SELECT COUNT(*) FROM card_names").fetchone()[0])
print("First row:", cursor.execute("SELECT * FROM card_names").fetchone())
print("Columns:", cursor.execute("SELECT * FROM card_names").description)

Done! Rows loaded: 102
First row: ('26000000', 'Knight')
Columns: (('team.card1.id', None, None, None, None, None, None), ('team.card1.name', None, None, None, None, None, None))
peak memory: 21211.87 MiB, increment: 4.21 MiB
CPU times: total: 2min 17s
Wall time: 3min 3s


In [6]:
%%time
%%memit
cursor.execute("""
    CREATE TABLE clasher_new AS
    SELECT c.*,
        w1."team.card1.name" as "winner.card1.name",
        w2."team.card1.name" as "winner.card2.name",
        w3."team.card1.name" as "winner.card3.name",
        w4."team.card1.name" as "winner.card4.name",
        w5."team.card1.name" as "winner.card5.name",
        w6."team.card1.name" as "winner.card6.name",
        w7."team.card1.name" as "winner.card7.name",
        w8."team.card1.name" as "winner.card8.name",
        l1."team.card1.name" as "loser.card1.name",
        l2."team.card1.name" as "loser.card2.name",
        l3."team.card1.name" as "loser.card3.name",
        l4."team.card1.name" as "loser.card4.name",
        l5."team.card1.name" as "loser.card5.name",
        l6."team.card1.name" as "loser.card6.name",
        l7."team.card1.name" as "loser.card7.name",
        l8."team.card1.name" as "loser.card8.name"
    FROM clasher c
    LEFT JOIN card_names w1 ON c."winner.card1.id" = w1."team.card1.id"
    LEFT JOIN card_names w2 ON c."winner.card2.id" = w2."team.card1.id"
    LEFT JOIN card_names w3 ON c."winner.card3.id" = w3."team.card1.id"
    LEFT JOIN card_names w4 ON c."winner.card4.id" = w4."team.card1.id"
    LEFT JOIN card_names w5 ON c."winner.card5.id" = w5."team.card1.id"
    LEFT JOIN card_names w6 ON c."winner.card6.id" = w6."team.card1.id"
    LEFT JOIN card_names w7 ON c."winner.card7.id" = w7."team.card1.id"
    LEFT JOIN card_names w8 ON c."winner.card8.id" = w8."team.card1.id"
    LEFT JOIN card_names l1 ON c."loser.card1.id" = l1."team.card1.id"
    LEFT JOIN card_names l2 ON c."loser.card2.id" = l2."team.card1.id"
    LEFT JOIN card_names l3 ON c."loser.card3.id" = l3."team.card1.id"
    LEFT JOIN card_names l4 ON c."loser.card4.id" = l4."team.card1.id"
    LEFT JOIN card_names l5 ON c."loser.card5.id" = l5."team.card1.id"
    LEFT JOIN card_names l6 ON c."loser.card6.id" = l6."team.card1.id"
    LEFT JOIN card_names l7 ON c."loser.card7.id" = l7."team.card1.id"
    LEFT JOIN card_names l8 ON c."loser.card8.id" = l8."team.card1.id"
""")

# 4. Replace clasher with the new table
cursor.execute("DROP TABLE clasher")
cursor.execute("ALTER TABLE clasher_new RENAME TO clasher")
conn.commit()

print("Done!", cursor.execute("SELECT COUNT(*) FROM clasher").fetchone()[0], "rows")
print("First row:", cursor.execute("SELECT * FROM clasher").fetchone())

Done! 16795959 rows
First row: ('0', '2020-12-07 07:00:00+00:00', '54000049.0', '72000201.0', '6590.0', '#28RR8PJP0', '6581.0', '31.0', '2.0', '4768.0', '[1218]', '#YYL2GJUC', '16000111.0', '#9UU8P008P', '6599.0', '-31.0', '1.0', '147.0', '#P2PUU9JV', '16000119.0', '', '', '26000036', '13', '28000015', '13', '26000050', '13', '26000044', '13', '26000054', '13', '28000016', '13', '26000043', '13', '26000062', '13', '[26000036, 26000043, 26000044, 26000050, 26000054, 26000062, 28000015, 28000016]', '104', '7', '0', '1', '1', '2', '3', '2', '3.625', '28000004', '13', '26000041', '13', '26000026', '13', '26000030', '13', '26000000', '13', '28000011', '13', '28000003', '13', '27000006', '13', '[26000000, 26000026, 26000030, 26000041, 27000006, 28000003, 28000004, 28000011]', '104', '4', '1', '3', '4', '1', '1', '2', '3.125', 'Battle Ram', 'Barbarian Barrel', 'Royal Ghost', 'Hunter', 'Cannon Cart', 'Heal Spirit', 'Elite Barbarians', 'Magic Archer', 'Goblin Barrel', 'Goblin Gang', 'Princess',

In [7]:
%%time
%%memit
cursor.execute("DROP TABLE IF EXISTS card_stats")
cursor.execute("""
    CREATE TABLE card_stats AS
    SELECT card_name, SUM(wins) as win_count, SUM(losses) as loss_count
    FROM (
        SELECT "winner.card1.name" as card_name, 1 as wins, 0 as losses FROM clasher UNION ALL
        SELECT "winner.card2.name", 1, 0 FROM clasher UNION ALL
        SELECT "winner.card3.name", 1, 0 FROM clasher UNION ALL
        SELECT "winner.card4.name", 1, 0 FROM clasher UNION ALL
        SELECT "winner.card5.name", 1, 0 FROM clasher UNION ALL
        SELECT "winner.card6.name", 1, 0 FROM clasher UNION ALL
        SELECT "winner.card7.name", 1, 0 FROM clasher UNION ALL
        SELECT "winner.card8.name", 1, 0 FROM clasher UNION ALL
        SELECT "loser.card1.name", 0, 1 FROM clasher UNION ALL
        SELECT "loser.card2.name", 0, 1 FROM clasher UNION ALL
        SELECT "loser.card3.name", 0, 1 FROM clasher UNION ALL
        SELECT "loser.card4.name", 0, 1 FROM clasher UNION ALL
        SELECT "loser.card5.name", 0, 1 FROM clasher UNION ALL
        SELECT "loser.card6.name", 0, 1 FROM clasher UNION ALL
        SELECT "loser.card7.name", 0, 1 FROM clasher UNION ALL
        SELECT "loser.card8.name", 0, 1 FROM clasher
    )
    WHERE card_name IS NOT NULL
    GROUP BY card_name
    ORDER BY win_count DESC
""")
conn.commit()

cursor.execute("SELECT * FROM card_stats LIMIT 10").fetchall()

peak memory: 250.52 MiB, increment: 21.39 MiB
CPU times: total: 12min 9s
Wall time: 12min 15s


In [8]:
%%time
%%memit
cursor.execute("ALTER TABLE card_stats ADD COLUMN win_ratio REAL")
cursor.execute("UPDATE card_stats SET win_ratio = ROUND(win_count * 1.0 / (win_count + loss_count), 3)")
conn.commit()

peak memory: 232.63 MiB, increment: 0.00 MiB
CPU times: total: 62.5 ms
Wall time: 1.1 s


In [9]:
results = cursor.execute("""
    SELECT * FROM card_stats 
    ORDER BY win_ratio DESC 
""").fetchall()

for row in results:
    print(row)

('Mega Minion', 966854, 898319, 0.518)
('Battle Ram', 509974, 476804, 0.517)
('Lava Hound', 311339, 292324, 0.516)
('Night Witch', 938355, 888178, 0.514)
('Golem', 1160228, 1102175, 0.513)
('Zap', 4798476, 4569975, 0.512)
('Lightning', 992210, 946387, 0.512)
('Goblin Gang', 2318752, 2216543, 0.511)
('P.E.K.K.A', 1941730, 1861776, 0.511)
('Flying Machine', 341577, 327391, 0.511)
('Bowler', 309926, 296324, 0.511)
('Rascals', 185359, 177601, 0.511)
('Mini P.E.K.K.A', 2705738, 2597622, 0.51)
('Dark Prince', 1193588, 1148216, 0.51)
('Goblin Barrel', 2845695, 2746123, 0.509)
('Skeleton Barrel', 1096119, 1056221, 0.509)
('Hog Rider', 3661370, 3547371, 0.508)
('Barbarian Barrel', 1262240, 1224167, 0.508)
('Cannon Cart', 266358, 258340, 0.508)
('Poison', 1179959, 1152655, 0.506)
('Guards', 556996, 543299, 0.506)
('Valkyrie', 5039353, 4931289, 0.505)
('Balloon', 2473828, 2421258, 0.505)
('Lumberjack', 1823562, 1786319, 0.505)
('Bandit', 1524480, 1495902, 0.505)
('Elite Barbarians', 1218512, 1195